# Stage 1: Data Preprocessing
Nhiệm vụ: Đọc file CSV thô, làm sạch dữ liệu, loại bỏ nhiễu do vung tay (Motion = 1) và xử lý giá trị NaN.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
import os

# Đọc dữ liệu của bạn
filepath = '../data/raw/PHU_30_PHUT_12-07-2026.csv'
df = pd.read_csv(filepath)

print(f"Tổng số dòng thu thập được: {len(df)} dòng")
df.head()

In [ ]:
# BƯỚC 1: LỌC NHIỄU CHUYỂN ĐỘNG (MOTION ARTIFACTS)
print("Số dòng bị gán cờ vung tay (Motion = 1):", len(df[df['Motion'] == 1]))

# Chỉ giữ lại những dòng bạn ngồi yên (Motion = 0)
df_clean = df[df['Motion'] == 0].copy()
df_clean.reset_index(drop=True, inplace=True)


In [ ]:
# BƯỚC 2: BANDPASS FILTER (Áp dụng cho TOÀN BỘ dữ liệu)
def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

fs = 100.0  
ir_signal_centered = df_clean['IR'] - df_clean['IR'].mean()
df_clean['IR_Filtered'] = butter_bandpass_filter(ir_signal_centered, 0.7, 4.0, fs, order=4)


In [ ]:
# BƯỚC 3: PHÁT HIỆN NHỊP TIM (PEAK DETECTION) TRÊN TOÀN BỘ DỮ LIỆU
# Giới hạn khoảng cách tối thiểu giữa 2 nhịp tim (ví dụ nhịp tim tối đa 200 BPM -> khoảng cách tối thiểu 300ms = 30 mẫu)
min_distance = int(0.3 * fs) 

# Tìm các đỉnh (peaks) trên tín hiệu đã lọc
peaks, _ = find_peaks(df_clean['IR_Filtered'], distance=min_distance, prominence=10)

print(f"Đã tìm thấy {len(peaks)} nhịp đập trong suốt quá trình đo!")

# VẼ BIỂU ĐỒ 10 GIÂY ĐỂ KIỂM CHỨNG BẰNG MẮT
start_idx = 200
end_idx = 1200

# Lọc ra các đỉnh nằm trong vùng vẽ biểu đồ
peaks_in_range = [p for p in peaks if start_idx <= p < end_idx]

plt.figure(figsize=(15, 4))
plt.plot(df_clean['Time(ms)'].iloc[start_idx:end_idx], df_clean['IR_Filtered'].iloc[start_idx:end_idx], color='blue', label='IR Signal')
plt.plot(df_clean['Time(ms)'].iloc[peaks_in_range], df_clean['IR_Filtered'].iloc[peaks_in_range], "ro", label='Heart Beats (Đỉnh)')

plt.title('Nhận diện nhịp tim tự động bằng AI/Toán học (Dấu chấm đỏ)')
plt.xlabel('Thời gian (ms)')
plt.ylabel('Biên độ IR')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# BƯỚC 4: LƯU DỮ LIỆU ĐÃ SẠCH ĐỂ DÙNG CHO BƯỚC TRÍCH XUẤT ĐẶC TRƯNG (FEATURE ENGINEERING)
# Tạo thư mục processed nếu chưa có
os.makedirs('../data/processed', exist_ok=True)

save_path = '../data/processed/clean_PHU_30_PHUT.csv'
df_clean.to_csv(save_path, index=False)
print(f"Đã lưu toàn bộ dữ liệu sạch (không có nhiễu vung tay, đã lọc sóng) vào: {save_path}")